## alt: zero shot labeling
Supervised version

In [1]:
pip install "scikit-learn<1.6" bertopic hdbscan umap-learn sentence-transformers konlpy pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\WINDOWS 11\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [2]:
from konlpy.tag import Okt
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

C:\Users\WINDOWS 11\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
BASE_DIR = Path("C:/Users/WINDOWS 11/Desktop/kpop_agenda/data/article")
INPUT_CSV = Path("C:/Users/WINDOWS 11/Desktop/kpop_agenda/data/top300_filtered.csv")
OUTPUT_CSV = Path("C:/Users/WINDOWS 11/Desktop/kpop_agenda/data/top300_filtered_with_topics_ZeroShot.csv")
ARTICLES_DIR = BASE_DIR

# Parameters (Adjust!!)
# UMAP_PARAMS = {
#     'n_neighbors': 8,
#     'n_components': 8,
#     'min_dist': 0.1,
#     'random_state': 42
# }

# HDBSCAN_PARAMS = {
#     'min_cluster_size': 20,
#     'min_samples': 4,
#     'cluster_selection_method': 'leaf'
# }

UMAP_PARAMS = {
    'n_neighbors': 8,
    'n_components': 8,
    'min_dist': 0.1,
    'random_state': 42
}

HDBSCAN_PARAMS = {
    'min_cluster_size': 27,
    'min_samples': 4,
    'cluster_selection_method': 'leaf'
}


In [26]:
def load_and_preprocess_documents(csv_path, articles_dir):
    """
    Loads metadata, reads article files, and tokenizes Korean text.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found at: {csv_path}")

    df = pd.read_csv(csv_path)
    okt = Okt()
    processed_docs = []
    valid_indices = []

    print("Processing documents...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Extract filename safely regardless of what path is in the CSV
        file_name = Path(row['file_path']).name
        full_path = articles_dir / file_name

        try:
            with open(full_path, 'r', encoding='utf-8') as f:
                text = f.read()
                
            # Tokenize using KoNLPy (Okt)
            tokens = okt.nouns(text)
            # Keep only nouns that are at least 2 characters long (removes single-letter noise)
            tokens = [word for word in tokens if len(word) > 1]
            processed_text = " ".join(tokens)
            
            processed_docs.append(processed_text)
            valid_indices.append(idx)
            
        except Exception as e:
            print(f"Warning: Could not read file {file_name}. Error: {e}")

    # Filter dataframe to match successfully read documents
    df_clean = df.loc[valid_indices].copy()
    return df_clean, processed_docs

In [27]:
from sklearn.feature_extraction.text import CountVectorizer

def run_topic_modeling():
    df, documents = load_and_preprocess_documents(INPUT_CSV, ARTICLES_DIR)
    documents = [str(doc) for doc in documents]

    zeroshot_topic_list = [
        "스타", 
        "드라마", 
        "예능", 
        "뮤직", 
        "영화"
    ]

    # ✨ FIX: Define words you want to explicitly ban from the topic keywords
    korean_stop_words = ['통해', '단독', '♥', '라며', '사진', '기자', '방송', '뉴스', '텐아시아', '배우', '사람', '스포츠조선', '엑스포츠', 'OSEN', '소식', '스타뉴스', '마이데일리']
    
    # Pass them into a CountVectorizer
    vectorizer_model = CountVectorizer(stop_words=korean_stop_words)

    embedding_model = SentenceTransformer("jhgan/ko-sbert-sts")
    embeddings = embedding_model.encode(documents, show_progress_bar=True, batch_size=4)

    umap_model = UMAP(**UMAP_PARAMS)
    hdbscan_model = HDBSCAN(**HDBSCAN_PARAMS)

    print("Fitting Zero-Shot BERTopic model...")
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        zeroshot_topic_list=zeroshot_topic_list,   
        zeroshot_min_similarity=0.75,
    )
    
    topics, probs = topic_model.fit_transform(documents, embeddings=embeddings)

    # 6. Save results to DataFrame
    df['topic'] = topics
    
    # Count topics (excluding -1 noise)
    unique_topics = set(topics)
    num_topics = len(unique_topics) - 1 if -1 in unique_topics else len(unique_topics)
    print(f"Success! Found {num_topics} topics.")

    # 7. Export and return
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Results saved to: {OUTPUT_CSV}")

    return topic_model, df

# Run the function
topic_model, final_df = run_topic_modeling()

# Immediately display the topic dataframe to verify the clean keywords
display(topic_model.get_topic_info())

Processing documents...


 52%|█████▏    | 155/300 [00:06<00:05, 28.76it/s]

 93%|█████████▎| 278/300 [00:11<00:00, 24.78it/s]

Batches: 100%|██████████| 75/75 [00:46<00:00,  1.63it/s]


Fitting Zero-Shot BERTopic model...
Success! Found 4 topics.
Results saved to: C:\Users\WINDOWS 11\Desktop\kpop_agenda\data\top300_filtered_with_topics_ZeroShot.csv


,Topic,Count,Name,Representation,Representative_Docs
0,-1,90,-1_공개_모습_조진웅_유튜브,"[공개, 모습, 조진웅, 유튜브, 아시아, 대상, 아이, 운명, 출연, 작품]",[전현무 올해 예능인 수상한 박나래 하차 혼자 산다 일련 논란 관해 사과 상암동 미...
1,0,78,0_박나래_매니저_주장_논란,"[박나래, 매니저, 주장, 논란, 의혹, 공개, 조세호, 사실, 입장, 관련]",[스타 뉴스 최혜진 기자 코미디언 박나래 사진 이동훈 코미디언 박나래 대한 자극 사...
2,1,58,1_결혼_최준희_결혼식_공개,"[결혼, 최준희, 결혼식, 공개, 아내, 김종국, 지난, 부부, 영상, 이혼]",[김종국 마이데일리 김진 기자 머리칼 하나 공개 지난해 결혼 김종국 꽁꽁 아내 정체...
3,2,45,2_이서진_웃음_멤버_아시아,"[이서진, 웃음, 멤버, 아시아, 김광규, 조인성, 김대호, 모습, 공개, 예능]",[아시아 유나 기자 김광규 김주하 만남 회상 핑크 분위기 지난 오후 방송 김주하 데...
4,3,27,3_선수_세상_권민아_가족,"[선수, 세상, 권민아, 가족, 지난, 병원, 사망, 마음, 원혁, 수영장]",[사진 권민아 스포츠서울 최승 기자 그룹 출신 배우 권민아 피부 시술 의료사 권민아...


In [28]:
print("Raw Counts from DataFrame")
topic_counts = final_df['topic'].value_counts().sort_values(ascending=False)

print(f"{'Topic ID':<10} | {'Article Count'}")
print("-" * 25)
for topic_id, count in topic_counts.items():
    print(f"{topic_id:<10} | {count}")

Raw Counts from DataFrame
Topic ID   | Article Count
-------------------------
-1         | 90
0          | 78
1          | 58
2          | 45
3          | 27


In [29]:
# This will output a beautifully formatted dataframe showing the topic names and keywords
topic_info = topic_model.get_topic_info()
display(topic_info)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,90,-1_공개_모습_조진웅_유튜브,"[공개, 모습, 조진웅, 유튜브, 아시아, 대상, 아이, 운명, 출연, 작품]",[전현무 올해 예능인 수상한 박나래 하차 혼자 산다 일련 논란 관해 사과 상암동 미...
1,0,78,0_박나래_매니저_주장_논란,"[박나래, 매니저, 주장, 논란, 의혹, 공개, 조세호, 사실, 입장, 관련]",[스타 뉴스 최혜진 기자 코미디언 박나래 사진 이동훈 코미디언 박나래 대한 자극 사...
2,1,58,1_결혼_최준희_결혼식_공개,"[결혼, 최준희, 결혼식, 공개, 아내, 김종국, 지난, 부부, 영상, 이혼]",[김종국 마이데일리 김진 기자 머리칼 하나 공개 지난해 결혼 김종국 꽁꽁 아내 정체...
3,2,45,2_이서진_웃음_멤버_아시아,"[이서진, 웃음, 멤버, 아시아, 김광규, 조인성, 김대호, 모습, 공개, 예능]",[아시아 유나 기자 김광규 김주하 만남 회상 핑크 분위기 지난 오후 방송 김주하 데...
4,3,27,3_선수_세상_권민아_가족,"[선수, 세상, 권민아, 가족, 지난, 병원, 사망, 마음, 원혁, 수영장]",[사진 권민아 스포츠서울 최승 기자 그룹 출신 배우 권민아 피부 시술 의료사 권민아...


In [24]:
# Define where to save the new Top 5 CSV
TOP5_CSV_PATH = Path("C:/Users/WINDOWS 11/Desktop/kpop_agenda/data/top5_articles_per_topic_ZeroShot.csv")

def save_top_5_by_views(df, output_path):
    print("Extracting the top 5 most viewed articles per topic...")
    
    # 1. Filter out the noise/outlier topic (-1)
    valid_topics = df[df['topic'] != -1].copy()

    # (Optional safeguard) Ensure 'views' is treated as a number, not text
    valid_topics['views'] = pd.to_numeric(valid_topics['views'], errors='coerce')

    # 2. Sort by 'topic' (ascending) and actual 'views' (descending), then take the top 5
    top5_df = (valid_topics
               .sort_values(by=['topic', 'views'], ascending=[True, False])
               .groupby('topic')
               .head(5))

    # 3. Save to CSV
    top5_df.to_csv(output_path, index=False)
    print(f"Success! Top 5 most viewed articles per topic saved to: {output_path}")
    
    # 4. Display a quick preview to verify 
    # (showing topic, views, title, and file_path so you can see the results instantly)
    display(top5_df[['topic', 'views', 'title', 'file_path']])

# Run the extractor
save_top_5_by_views(final_df, TOP5_CSV_PATH)

Extracting the top 5 most viewed articles per topic...
Success! Top 5 most viewed articles per topic saved to: C:\Users\WINDOWS 11\Desktop\kpop_agenda\data\top5_articles_per_topic_ZeroShot.csv


,topic,views,title,file_path
0,0,1823020,"신기루, 법정 구속됐다…""혐의 인정 못해"" 결국 상습 허언죄로 입소 ('배불리힐스')",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
4,0,1484542,"유재석, 결국 비난 쏟아냈다…""정말 최악, 줘도 안 가져"" 허접한 비밀 선물에 격분...",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
6,0,1465801,"[단독]""매니저도 안돼…"" 김종국, 강남 J호텔서 결혼식 마쳐",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
10,0,1406119,"샘 해밍턴 9살 아들 벤틀리 ""난 한국인 아닌 호주 사람""...정체성 벌써 정했다",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
13,0,1374750,"김대호, 제대로 사고쳤다…달동네 2억집 싹 고쳤는데 ""20박스 모래 부어"" ('나혼산')",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
1,1,1809646,"박나래 논란, 새 폭로로 ‘국면 전환’… 전 매니저 “잠든 박나래에 주사이모 계속 ...",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
2,1,1586302,"이진호 ""박나래 매니저들, 55억 이태원 자택 도둑 사건後 큰 배신감…폭로 촉발 결...",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
3,1,1527854,"""성관계 있었지만 억지로 한 것 아니다"" 소속 배우 '성폭행 혐의' 체포된 기획사 대표",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
7,1,1449491,"“소리 지르며 촬영장 이탈”.. 박나래, 논란 속 ‘놀토’서 돌발 해프닝 ('놀토')",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
9,1,1409302,"""모두 진짜였다"" 이이경 사생활 루머 폭로자, 계정 삭제…'극심한 피해' 어쩌나 [...",C:/Users/WINDOWS 11/Desktop/kpop_agenda/data\2...
